# Advanced MCP Client: Raw SDK + Multi-Step Shopping Flow

The existing `client.py` wraps MCP tools inside a LangChain agent — the agent decides which tools to call and when. This notebook goes one level lower: it uses the **raw MCP SDK** (`ClientSession`) to call tools *explicitly*, with no agent in the loop.

Goals:
1. Connect to the cat shop server over **Streamable HTTP**
2. Authenticate via **OAuth 2.0 + PKCE** (same flow as `client.py`, reusing its helpers)
3. Orchestrate an explicit **browse → add to cart → checkout** shopping flow
4. Compare the developer experience of **MCP vs traditional REST**

> **Before running**: make sure `uv run server.py` is running in another terminal.

In [14]:
import asyncio
import json
import os
import webbrowser
from contextlib import AsyncExitStack

import httpx
from dotenv import load_dotenv
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client
from mcp.client.auth import OAuthClientProvider
from mcp.shared.auth import OAuthClientMetadata
from pydantic import AnyUrl

from client import InMemoryTokenStorage, OAuthCallbackServer

load_dotenv()
SERVER_URL = os.getenv("MCP_SERVER_URL", "http://localhost:8000")
MCP_URL = f"{SERVER_URL}/mcp"
print(f"Targeting MCP server at: {MCP_URL}")

Targeting MCP server at: http://localhost:8000/mcp


## OAuth: Why It's Needed Here

The cat shop server runs as a public HTTP service — not a local subprocess. Any network-accessible server needs auth to:
- Know *who* is making each request (so carts are per-user)
- Prevent unauthenticated callers from reaching tools

The flow is **OAuth 2.0 Authorization Code + PKCE**:
1. Client registers itself with the server (dynamic client registration)
2. Client opens a browser to the server's login page
3. User enters a username → server issues an auth code
4. Client exchanges the code (+ PKCE verifier) for an access token
5. All subsequent MCP calls carry the token as a Bearer header

**`OAuthClientProvider`** implements `httpx.Auth`, so it plugs directly into an `httpx.AsyncClient`. The SDK then handles token refresh automatically.

In [32]:
# Close any existing stack from a previous run before re-entering.
# Swallow the benign anyio cancel-scope error (see teardown cell for why).
if 'stack' in dir() and stack is not None:
    try:
        await stack.aclose()
    except BaseException:
        pass
    stack = None

# AsyncExitStack lets us hold context managers open across multiple notebook cells.
# A plain `async with` would close the connection at the end of this cell.
stack = AsyncExitStack()

# Start the local callback server that catches the OAuth redirect
callback_server = await stack.enter_async_context(OAuthCallbackServer(port=8765))

async def redirect_handler(url: str) -> None:
    print(f"\nOpen this URL to sign in:\n  {url}\n")
    webbrowser.open(url)

oauth = OAuthClientProvider(
    server_url=SERVER_URL,
    client_metadata=OAuthClientMetadata(
        client_name="Cat Shop Advanced Client",
        redirect_uris=[AnyUrl(callback_server.callback_url)],
        grant_types=["authorization_code", "refresh_token"],
        response_types=["code"],
        scope="read write",
    ),
    storage=InMemoryTokenStorage(),
    redirect_handler=redirect_handler,
    callback_handler=callback_server.wait_for_callback,
)

# Wire OAuth into httpx, then pass that client to the MCP transport
http_client = httpx.AsyncClient(auth=oauth)
read, write, _ = await stack.enter_async_context(
    streamable_http_client(MCP_URL, http_client=http_client)
)
session = await stack.enter_async_context(ClientSession(read, write))
await session.initialize()
print("Connected and authenticated.")


Open this URL to sign in:
  http://localhost:8000/authorize?response_type=code&client_id=b860c938-927d-480e-9fd8-f3abbdc19c18&redirect_uri=http%3A%2F%2F127.0.0.1%3A8765%2Fcallback&state=Oa7pi-i_eZ_V2M_uz4x8LH8OvvHqYEtFydVqcTzFKJ8&code_challenge=C1wTHb2EI3QkCCLnRmsbSPUpgGUZUeQOWYL9oJ2HxH0&code_challenge_method=S256&resource=http%3A%2F%2Flocalhost%3A8000%2F&scope=read+write

Connected and authenticated.


In [33]:
# Helper: call a tool by name, return parsed Python object.
# Prefer structuredContent: it holds the tool's FULL return value. FastMCP
# wraps non-dict returns (e.g. a list) under a "result" key.
#
# The trap: when a tool returns a list, the protocol emits ONE content block
# per element — so content[0].text is just the FIRST item, not the whole list.
# Parsing only content[0] then iterating gives you a single dict's string keys.
async def call(tool_name: str, **kwargs):
    result = await session.call_tool(tool_name, kwargs)
    if result.isError:
        raise RuntimeError(result.content[0].text if result.content else "Tool error")
    sc = result.structuredContent
    if sc is not None:
        # FastMCP wraps non-dict returns as {"result": <value>}
        if isinstance(sc, dict) and set(sc.keys()) == {"result"}:
            return sc["result"]
        return sc
    return json.loads(result.content[0].text)

print("call() helper ready.")

call() helper ready.


## Step 1: Discover Available Tools

Unlike REST (where you need docs to know what endpoints exist), MCP has **in-protocol tool discovery** — `list_tools()` returns every tool's name, description, and typed input schema. An AI agent can use this to decide which tools to call without any prior knowledge.

In [34]:
tools_result = await session.list_tools()
print(f"Server exposes {len(tools_result.tools)} tools:\n")
for tool in tools_result.tools:
    params = list(tool.inputSchema.get("properties", {}).keys())
    param_str = f"({', '.join(params)})" if params else "()"
    print(f"  {tool.name}{param_str}")
    print(f"    {tool.description}")

Server exposes 7 tools:

  list_products(category)
    Browse the cat shop catalog. Optionally filter by category (toys, beds, food, furniture).
  search_products(query)
    Search the cat shop catalog by name or description. Returns matching products.
  get_product(product_id)
    Get full details of a single product by its ID.
  add_to_cart(product_id, quantity)
    Add a product to your shopping cart. If already in cart, quantity is increased.
  view_cart()
    View everything in your shopping cart with quantities and totals.
  remove_from_cart(product_id)
    Remove a product from your shopping cart.
  checkout()
    Complete your purchase. Shows order summary and clears the cart.


## Step 2: Browse the Catalog

In [35]:
products = await call("list_products")
print(f"Found {len(products)} products:\n")
for p in products:
    print(f"  [{p['id']}] {p['name']:<30} ${p['price']:>6.2f}  ({p['category']})")

Found 8 products:

  [1] Whisker Wand                   $  9.99  (toys)
  [2] Catnip Mouse                   $  4.99  (toys)
  [3] Laser Pointer Pro              $ 12.99  (toys)
  [4] Cozy Cat Bed                   $ 29.99  (beds)
  [5] Window Hammock                 $ 24.99  (beds)
  [6] Salmon Treats                  $  7.99  (food)
  [7] Tuna Crunchies                 $  5.99  (food)
  [8] Scratching Post Tower          $ 49.99  (furniture)


## Step 3: Search for Something Specific

In [36]:
results = await call("search_products", query="salmon")
print(f"Search 'salmon' → {len(results)} result(s):")
for p in results:
    print(f"  [{p['id']}] {p['name']} — {p['description']}")

Search 'salmon' → 1 result(s):
  [6] Salmon Treats — Freeze-dried wild salmon bites, 100g


## Step 4: Get Product Detail

In [37]:
chosen_id = products[0]["id"]
detail = await call("get_product", product_id=chosen_id)
print(f"Product detail for ID {chosen_id}:")
for k, v in detail.items():
    print(f"  {k}: {v}")

Product detail for ID 1:
  id: 1
  name: Whisker Wand
  description: Interactive feather toy on a flexible wand
  price: 9.99
  category: toys


## Step 5: Add Items to Cart

In [38]:
r1 = await call("add_to_cart", product_id=products[0]["id"], quantity=2)
print(r1["message"])

r2 = await call("add_to_cart", product_id=products[1]["id"], quantity=1)
print(r2["message"])

Added 2x Whisker Wand to your cart
Added 1x Catnip Mouse to your cart


## Step 6: View Cart

In [39]:
cart = await call("view_cart")
print(f"Cart ({cart['item_count']} item(s)):\n")
for item in cart["items"]:
    print(f"  {item['name']:<30} x{item['quantity']}  ${item['subtotal']:.2f}")
print(f"\n  {'TOTAL':<35} ${cart['total']:.2f}")

Cart (2 item(s)):

  Whisker Wand                   x2  $19.98
  Catnip Mouse                   x1  $4.99

  TOTAL                               $24.97


## Step 7: Checkout

In [40]:
order = await call("checkout")
print(f"Order {order['order_id']} — {order['status'].upper()}")
print(f"Total: ${order['total']:.2f}")
print(f"\n{order['message']}")

Order 5E733D4B2B1B597B — CONFIRMED
Total: $24.97

Order 5E733D4B2B1B597B confirmed! Thanks reuben, your cats will love their new goodies!


In [41]:
# Teardown: close session, transport, and callback server.
#
# Heads-up: the MCP transport uses anyio cancel scopes that MUST be exited in
# the SAME asyncio task that entered them. Jupyter may run each cell in a
# *different* task, so closing the stack here (a later cell than where it was
# opened) can raise:
#   "Attempted to exit cancel scope in a different task than it was entered in"
# This is benign at shutdown — aclose() still unwinds every context manager
# (httpx client + callback server get closed); only anyio's cancel-scope
# bookkeeping complains. We swallow that one error and re-raise anything else.
def _all_benign(exc):
    if isinstance(exc, BaseExceptionGroup):
        return all(_all_benign(e) for e in exc.exceptions)
    return isinstance(exc, RuntimeError) and "cancel scope" in str(exc)

try:
    await stack.aclose()
    print("Session closed.")
except BaseException as e:
    if _all_benign(e):
        print("Session closed (ignored benign anyio cancel-scope error from closing across cells).")
    else:
        raise
finally:
    stack = None

Session closed (ignored benign anyio cancel-scope error from closing across cells).


---

## MCP vs Traditional REST: Developer Experience Comparison

The same shopping flow written against a hypothetical REST API:

### Discovery

**MCP** — in-protocol, machine-readable:
```python
tools = await session.list_tools()
# returns typed Tool objects with name, description, inputSchema
```

**REST** — out-of-band, manual:
```python
# Read OpenAPI docs, reverse-engineer URLs, no in-band schema
# GET /products, POST /cart/items, POST /checkout — you just have to know these
```

---

### Authentication

**MCP** — wired once at transport level; SDK manages token refresh:
```python
oauth = OAuthClientProvider(server_url=..., storage=..., redirect_handler=..., ...)
http_client = httpx.AsyncClient(auth=oauth)  # done — all calls are authenticated
```

**REST** — manual Bearer header on every call; developer manages refresh:
```python
token = await get_token()  # your problem
headers = {"Authorization": f"Bearer {token}"}
resp = await client.get("/products", headers=headers)
if resp.status_code == 401:  # also your problem
    token = await refresh_token()
    ...
```

---

### Making a Call

**MCP** — uniform across all tools, but requires unwrapping the envelope:
```python
result = await session.call_tool("list_products", {})
products = json.loads(result.content[0].text)  # extra parse step
```

**REST** — direct JSON, less boilerplate for a known endpoint:
```python
resp = await client.get("/products")
resp.raise_for_status()
products = resp.json()  # direct
```

---

### Adding to Cart

**MCP:**
```python
await session.call_tool("add_to_cart", {"product_id": 1, "quantity": 2})
```

**REST:**
```python
await client.post("/cart/items", json={"product_id": 1, "quantity": 2})
```

---

### Error Handling

**MCP** — uniform `isError` flag across all tools:
```python
result = await session.call_tool("checkout", {})
if result.isError:
    print(result.content[0].text)  # same shape regardless of which tool failed
```

**REST** — HTTP status codes + inconsistent error bodies per API:
```python
resp = await client.post("/checkout")
if resp.status_code == 400:
    error = resp.json()  # shape varies — could be {"error": ...}, {"message": ...}, etc.
```

---

## DX Analysis

| Dimension | MCP | Traditional REST |
|---|---|---|
| Tool discovery | In-protocol JSON Schema via `list_tools()` | Out-of-band docs (OpenAPI, README) |
| Auth lifecycle | SDK-managed; token refresh automatic | Manual Bearer header per call; refresh DIY |
| Response parsing | Extra `json.loads(content[0].text)` unwrap | Direct `.json()` |
| Response shape | Always `CallToolResult` — uniform | Varies per endpoint and API designer |
| Type safety | `inputSchema` enables pre-call validation | Bespoke per endpoint |
| Multi-step session | Stateful; auth context maintained across calls | Stateless HTTP; auth repeated |
| AI agent integration | First-class — `list_tools()` → `call_tool()` | Requires custom wrappers and prompt engineering |
| Setup complexity | ~40 lines of OAuth boilerplate | `httpx.AsyncClient(base_url=..., headers=...)` |

### Conclusion

MCP adds a protocol envelope and OAuth plumbing that REST doesn't need when you're calling known endpoints from application code. For a developer who already knows the API, `httpx.get("/products")` is objectively simpler than the MCP equivalent.

The payoff shows up when an **AI agent** is the caller:
- `list_tools()` gives the agent a complete, typed, machine-readable capability map at runtime — no prior knowledge needed
- `inputSchema` lets the agent (or a validation layer) check its arguments before calling
- The uniform `CallToolResult` envelope means the agent's error-handling logic doesn't need to know anything about the specific tool
- Transport-level OAuth means every tool is automatically authenticated with the right user context

REST is the right choice when you control both client and server and know the endpoints statically. MCP is the right choice when an AI agent needs to discover and call capabilities dynamically at runtime — which is exactly the use case it was designed for.